In [ ]:
# BAN6800 – BA-07: Feature Engineering and Validation
# CGSL Predictive Maintenance Project

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Feature engineering libraries loaded successfully.")

Feature engineering libraries loaded successfully.


In [ ]:
# Load processed feature arrays from GitHub LFS

import requests
from io import BytesIO

train_features_url = "https://media.githubusercontent.com/media/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/processed/X_train_imputed.npy"
test_features_url = "https://media.githubusercontent.com/media/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/processed/X_test_imputed.npy"

train_response = requests.get(train_features_url)
train_response.raise_for_status()

test_response = requests.get(test_features_url)
test_response.raise_for_status()

X_train = np.load(
    BytesIO(train_response.content),
    allow_pickle=False
)

X_test = np.load(
    BytesIO(test_response.content),
    allow_pickle=False
)

print("Processed feature arrays loaded successfully.")
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Processed feature arrays loaded successfully.
Training shape: (60000, 335)
Test shape: (16000, 335)


In [ ]:
# Load processed target labels

y_train_url = "https://raw.githubusercontent.com/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/processed/y_train.csv"
y_test_url = "https://raw.githubusercontent.com/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/processed/y_test.csv"

y_train = pd.read_csv(y_train_url).squeeze("columns")
y_test = pd.read_csv(y_test_url).squeeze("columns")

print("Training target shape:", y_train.shape)
print("Test target shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

Training target shape: (60000,)
Test target shape: (16000,)

Training target distribution:
class
neg    59000
pos     1000
Name: count, dtype: int64


In [ ]:
# Check for zero-variance features after preprocessing

feature_variances = np.var(X_train, axis=0)

zero_variance_indices = np.where(feature_variances == 0)[0]

print("Total transformed features:", X_train.shape[1])
print("Zero-variance features:", len(zero_variance_indices))

if len(zero_variance_indices) > 0:
    print("Indices of zero-variance features:")
    print(zero_variance_indices)
else:
    print("No zero-variance features found.")

Total transformed features: 335
Zero-variance features: 1
Indices of zero-variance features:
[87]


In [ ]:
# Remove zero-variance features

non_constant_indices = np.where(feature_variances > 0)[0]

X_train_selected = X_train[:, non_constant_indices]
X_test_selected = X_test[:, non_constant_indices]

print("Original feature count:", X_train.shape[1])
print("Zero-variance features removed:", len(zero_variance_indices))
print("Final training feature count:", X_train_selected.shape[1])
print("Final test feature count:", X_test_selected.shape[1])

Original feature count: 335
Zero-variance features removed: 1
Final training feature count: 334
Final test feature count: 334


## BA-07 Finding: Zero-Variance Feature

The transformed training matrix contained 335 features. One feature had zero variance across all training observations and therefore provided no discriminatory information.

The zero-variance feature was removed from both the training and test matrices using the same feature index to maintain structural consistency. This reduced the modelling feature set from 335 to 334 features.

The feature was removed based on training-data characteristics only.

In [ ]:
# Check for highly correlated features using training data only

# Convert the training array to a DataFrame
X_train_df = pd.DataFrame(X_train_selected)

# Calculate the absolute correlation matrix
corr_matrix = X_train_df.corr().abs()

# Keep only the upper triangle to avoid duplicate feature pairs
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Identify feature pairs with correlation >= 0.95
high_corr_pairs = []

for col in upper_triangle.columns:
    correlated_features = upper_triangle.index[
        upper_triangle[col] >= 0.95
    ].tolist()

    for row in correlated_features:
        high_corr_pairs.append(
            (row, col, upper_triangle.loc[row, col])
        )

high_corr_df = pd.DataFrame(
    high_corr_pairs,
    columns=["Feature_1", "Feature_2", "Absolute_Correlation"]
).sort_values(
    "Absolute_Correlation",
    ascending=False
)

print("Number of highly correlated feature pairs (>= 0.95):",
      len(high_corr_df))

display(high_corr_df.head(20))

Number of highly correlated feature pairs (>= 0.95): 3020


,Feature_1,Feature_2,Absolute_Correlation
3002,249,333,1.0
1759,277,283,1.0
1760,278,283,1.0
1761,279,283,1.0
1762,280,283,1.0
1763,281,283,1.0
1764,282,283,1.0
50,170,171,1.0
51,172,173,1.0
52,172,174,1.0


In [ ]:
# Recover feature names from the preprocessing imputer

original_feature_names = X_train_df.columns

try:
    feature_names = imputer.get_feature_names_out()
    print("Number of imputer-generated feature names:", len(feature_names))
    print("\nFirst 20 feature names:")
    print(feature_names[:20])
except NameError:
    print("The imputer object is not available in this notebook.")
    print("We will recover feature names from the BA-05 preprocessing notebook.")

The imputer object is not available in this notebook.
We will recover feature names from the BA-05 preprocessing notebook.


In [ ]:
# Recreate the BA-05 preprocessing structure to recover exact feature names

from sklearn.impute import SimpleImputer

train_url = "https://raw.githubusercontent.com/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/raw/aps_failure_training_set.csv"

df_raw = pd.read_csv(
    train_url,
    skiprows=20,
    na_values="na"
)

# Separate target from features
X_raw = df_raw.drop(columns=["class"])

# Use the same >80% missingness rule from BA-05
missing_rate = X_raw.isna().mean()
high_missing_features = missing_rate[missing_rate > 0.80].index.tolist()

X_raw_reduced = X_raw.drop(columns=high_missing_features)

# Recreate the same imputer
imputer_rebuild = SimpleImputer(
    strategy="median",
    add_indicator=True
)

X_rebuilt = imputer_rebuild.fit_transform(X_raw_reduced)

# Recover exact transformed feature names
feature_names = imputer_rebuild.get_feature_names_out(
    X_raw_reduced.columns
)

print("Features after high-missingness removal:", X_raw_reduced.shape[1])
print("Transformed feature count:", len(feature_names))
print("Rebuilt matrix shape:", X_rebuilt.shape)

Features after high-missingness removal: 168
Transformed feature count: 335
Rebuilt matrix shape: (60000, 335)


In [14]:
# Verify that the rebuilt preprocessing dimensions match the saved arrays

print("Saved X_train shape:", X_train.shape)
print("Rebuilt X_train shape:", X_rebuilt.shape)

print("Feature-count match:",
      X_train.shape[1] == X_rebuilt.shape[1])

print("\nFirst 30 transformed feature names:")
print(feature_names[:30])

Saved X_train shape: (60000, 335)
Rebuilt X_train shape: (60000, 335)
Feature-count match: True

First 30 transformed feature names:
['aa_000' 'ab_000' 'ac_000' 'ad_000' 'ae_000' 'af_000' 'ag_000' 'ag_001'
 'ag_002' 'ag_003' 'ag_004' 'ag_005' 'ag_006' 'ag_007' 'ag_008' 'ag_009'
 'ah_000' 'ai_000' 'aj_000' 'ak_000' 'al_000' 'am_0' 'an_000' 'ao_000'
 'ap_000' 'aq_000' 'ar_000' 'as_000' 'at_000' 'au_000']


In [15]:
# Identify the exact feature corresponding to the zero-variance column

zero_variance_index = 87

print("Zero-variance column index:", zero_variance_index)
print("Zero-variance feature name:", feature_names[zero_variance_index])

Zero-variance column index: 87
Zero-variance feature name: cd_000


In [16]:
# Remove the identified zero-variance feature

zero_variance_index = 87

X_train_final = np.delete(X_train_selected, zero_variance_index, axis=1)
X_test_final = np.delete(X_test_selected, zero_variance_index, axis=1)

feature_names_final = np.delete(feature_names, zero_variance_index)

print("Removed feature:", feature_names[zero_variance_index])
print("Final training shape:", X_train_final.shape)
print("Final test shape:", X_test_final.shape)
print("Final feature-name count:", len(feature_names_final))

Removed feature: cd_000
Final training shape: (60000, 333)
Final test shape: (16000, 333)
Final feature-name count: 334


In [17]:
# Save the final modelling feature-name mapping

feature_mapping = pd.DataFrame({
    "Model_Feature_Index": range(len(feature_names_final)),
    "Feature_Name": feature_names_final
})

feature_mapping.to_csv(
    "BA-07_final_feature_mapping.csv",
    index=False
)

print("Final feature mapping saved successfully.")

Final feature mapping saved successfully.


In [18]:
# Save the final feature matrices

np.save("X_train_final.npy", X_train_final)
np.save("X_test_final.npy", X_test_final)

print("Final feature matrices saved successfully.")

Final feature matrices saved successfully.


In [21]:
# Correct the final feature set after removing the zero-variance feature

X_train_final = X_train_selected
X_test_final = X_test_selected

# Keep only the feature names that survived the zero-variance removal
feature_names_final = feature_names[non_constant_indices]

print("Removed zero-variance feature:", feature_names[zero_variance_indices[0]])
print("Final training shape:", X_train_final.shape)
print("Final test shape:", X_test_final.shape)
print("Final feature-name count:", len(feature_names_final))

Removed zero-variance feature: cd_000
Final training shape: (60000, 334)
Final test shape: (16000, 334)
Final feature-name count: 334


In [22]:
# Create the corrected final feature mapping

feature_mapping = pd.DataFrame({
    "Model_Feature_Index": range(len(feature_names_final)),
    "Feature_Name": feature_names_final
})

feature_mapping.to_csv(
    "BA-07_final_feature_mapping.csv",
    index=False
)

print("Feature mapping shape:", feature_mapping.shape)
print("\nFirst 10 final features:")
display(feature_mapping.head(10))

Feature mapping shape: (334, 2)

First 10 final features:


,Model_Feature_Index,Feature_Name
0,0,aa_000
1,1,ab_000
2,2,ac_000
3,3,ad_000
4,4,ae_000
5,5,af_000
6,6,ag_000
7,7,ag_001
8,8,ag_002
9,9,ag_003


In [23]:
# Final consistency check

assert X_train_final.shape[1] == len(feature_names_final)
assert X_test_final.shape[1] == len(feature_names_final)

print("Feature mapping and matrices are consistent.")
print("Final modelling features:", len(feature_names_final))

Feature mapping and matrices are consistent.
Final modelling features: 334


In [24]:
# Assess feature redundancy using the training data only

high_corr_threshold = 0.95

X_final_df = pd.DataFrame(
    X_train_final,
    columns=feature_names_final
)

corr_matrix = X_final_df.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={
        "level_0": "Feature_1",
        "level_1": "Feature_2",
        0: "Absolute_Correlation"
    })
)

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["Absolute_Correlation"] >= high_corr_threshold
].sort_values(
    "Absolute_Correlation",
    ascending=False
)

print("Highly correlated pairs (>= 0.95):",
      len(high_corr_pairs))

print(
    "Perfectly correlated pairs (= 1.0):",
    (high_corr_pairs["Absolute_Correlation"] == 1.0).sum()
)

display(high_corr_pairs.head(20))

Highly correlated pairs (>= 0.95): 3020
Perfectly correlated pairs (= 1.0): 731


,Feature_1,Feature_2,Absolute_Correlation
55576,missingindicator_ee_003,missingindicator_ee_005,1.0
44286,missingindicator_ai_000,missingindicator_aj_000,1.0
44289,missingindicator_ai_000,missingindicator_am_0,1.0
44295,missingindicator_ai_000,missingindicator_as_000,1.0
44296,missingindicator_ai_000,missingindicator_at_000,1.0
44297,missingindicator_ai_000,missingindicator_au_000,1.0
44438,missingindicator_aj_000,missingindicator_am_0,1.0
44444,missingindicator_aj_000,missingindicator_as_000,1.0
44445,missingindicator_aj_000,missingindicator_at_000,1.0
44446,missingindicator_aj_000,missingindicator_au_000,1.0


In [25]:
# Identify how many features participate in high-correlation relationships

high_corr_features = pd.unique(
    high_corr_pairs[["Feature_1", "Feature_2"]].values.ravel()
)

print(
    "Unique features involved in >= 0.95 correlations:",
    len(high_corr_features)
)

print(
    "Percentage of final features involved:",
    round(
        len(high_corr_features) / len(feature_names_final) * 100,
        2
    ),
    "%"
)

Unique features involved in >= 0.95 correlations: 191
Percentage of final features involved: 57.19 %


## BA-07 Feature Redundancy Assessment and Decision

Correlation analysis was performed using the training data only on the 334-feature matrix remaining after zero-variance filtering.

The analysis identified 3,020 feature pairs with absolute correlation of at least 0.95. Of these, 731 pairs had perfect correlation of 1.0. A total of 191 of the 334 features (57.19%) participated in at least one high-correlation relationship.

Inspection of the perfectly correlated relationships showed that many involved missingness-indicator features created during the preprocessing stage. Therefore, high correlation does not automatically imply that the associated feature should be removed.

A conservative feature-selection strategy was adopted. The single zero-variance feature (`cd_000`) was removed, but no additional correlation-based pruning was performed at this stage. The resulting 334-feature matrix will be carried into baseline modelling, where model performance and validation results will provide evidence for any future dimensionality-reduction decision.

This approach avoids arbitrary information loss and keeps the feature-engineering process reproducible and data-driven.

In [26]:
# Save the final BA-07 feature set and feature mapping

np.save("X_train_final.npy", X_train_final)
np.save("X_test_final.npy", X_test_final)

feature_mapping.to_csv(
    "BA-07_final_feature_mapping.csv",
    index=False
)

print("Final BA-07 artifacts saved successfully.")
print("X_train_final:", X_train_final.shape)
print("X_test_final:", X_test_final.shape)
print("Feature mapping:", feature_mapping.shape)

Final BA-07 artifacts saved successfully.
X_train_final: (60000, 334)
X_test_final: (16000, 334)
Feature mapping: (334, 2)


In [27]:
# Save the correlation screening results

high_corr_pairs.to_csv(
    "BA-07_high_correlation_pairs.csv",
    index=False
)

print("Correlation screening report saved successfully.")
print("High-correlation pairs:", len(high_corr_pairs))

Correlation screening report saved successfully.
High-correlation pairs: 3020


In [28]:
from google.colab import files

files.download("BA-07_final_feature_mapping.csv")
files.download("BA-07_high_correlation_pairs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>